# EXP27: 宏观情景压力测试

用历史中类似宏观冲击的时间段做代理，测试当前实盘策略在 6 种极端情景下的表现。

| 情景 | 代理时段 | 市场特征 |
|---|---|---|
| S1 伊朗战争升级 | 2020-01~02 | 避险脉冲上涨 |
| S2 伊朗停火暴跌 | 2026-03-15~31 | 避险退潮单边暴跌 |
| S3 流动性危机 | 2020-03-09~23 | 股金齐跌 V型反转 |
| S4 央行抛售阴跌 | 2022-09~11 | 结构性卖压慢跌 |
| S5 央行囤积慢牛 | 2024-02~04 | 稳步上涨低波动 |
| S6 关税双向鞭打 | 2025-04-02~09 | 极端双向波动 |

In [ ]:
import sys; sys.path.insert(0, '..')
import json
import pandas as pd
import numpy as np
from backtest import DataBundle, run_variant, C12_KWARGS, V3_REGIME
from backtest.stats import calc_stats

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}

print('=== Full-period baseline ===')
full_baseline = run_variant(data, 'Full_Baseline', **LIVE_KWARGS)
print(f"N={full_baseline['n']}, Sharpe={full_baseline['sharpe']:.2f}, "
      f"PnL=${full_baseline['total_pnl']:.0f}, MaxDD=${full_baseline['max_dd']:.0f}")

## 情景定义与工具函数

In [ ]:
SCENARIOS = [
    {
        'id': 'S1', 'name': 'Iran Escalation (Risk-On Gold)',
        'start': '2020-01-01', 'end': '2020-02-28',
        'desc': 'US-Iran conflict escalation, safe-haven gold rally, VIX spike',
        'relevance': 'If Iran war escalates next week, gold surges on safe-haven demand',
    },
    {
        'id': 'S2', 'name': 'Iran Ceasefire (Risk-Off Gold Crash)',
        'start': '2026-03-15', 'end': '2026-03-31',
        'desc': 'Trump signals war ending, gold crashes -14.6% in 2 weeks',
        'relevance': 'If ceasefire announced, safe-haven premium evaporates rapidly',
    },
    {
        'id': 'S3', 'name': 'Liquidity Crisis (Stocks & Gold Down)',
        'start': '2020-03-09', 'end': '2020-03-23',
        'desc': 'COVID liquidity crunch, margin calls, gold drops then V-reversal',
        'relevance': 'If war + tariffs trigger margin calls, everything sells off together',
    },
    {
        'id': 'S4', 'name': 'Central Bank Selling (Structural Bear)',
        'start': '2022-09-01', 'end': '2022-11-30',
        'desc': 'Fed aggressive hikes, DXY at 114, gold slow bleed $1800->$1620',
        'relevance': 'Turkey sold 58t in March; if more CBs sell to defend currencies',
    },
    {
        'id': 'S5', 'name': 'Central Bank Buying (Structural Bull)',
        'start': '2024-02-01', 'end': '2024-04-30',
        'desc': 'China CB buying + rate cut expectations, gold $2000->$2400 (+20%)',
        'relevance': 'If CBs resume accumulation post-war, steady uptrend',
    },
    {
        'id': 'S6', 'name': 'Tariff Whipsaw (Extreme Two-Way Vol)',
        'start': '2025-04-01', 'end': '2025-04-15',
        'desc': 'Liberation Day tariffs, +3% one day -4% next, no clear direction',
        'relevance': 'New pharma/metal tariffs announced Apr 2, market chaos',
    },
]

def analyze_trades(trades, label=''):
    """Detailed breakdown of a trade list."""
    if not trades:
        return {'label': label, 'n': 0, 'pnl': 0, 'buy_n': 0, 'sell_n': 0}
    buy_trades = [t for t in trades if t.direction == 'BUY']
    sell_trades = [t for t in trades if t.direction == 'SELL']
    keltner = [t for t in trades if t.strategy == 'keltner']
    orb = [t for t in trades if t.strategy == 'orb']
    rsi = [t for t in trades if t.strategy == 'm15_rsi']
    
    wins = [t for t in trades if t.pnl > 0]
    trailing = [t for t in trades if 'Trailing' in str(t.exit_reason) or 'trailing' in str(t.exit_reason)]
    sl_exits = [t for t in trades if 'SL' in str(t.exit_reason) or 'stop' in str(t.exit_reason).lower()]
    timeout = [t for t in trades if 'timeout' in str(t.exit_reason).lower() or 'max_hold' in str(t.exit_reason).lower()]
    
    return {
        'label': label,
        'n': len(trades),
        'pnl': sum(t.pnl for t in trades),
        'wr': len(wins) / len(trades) * 100,
        'avg_pnl': np.mean([t.pnl for t in trades]),
        'avg_bars': np.mean([t.bars_held for t in trades]),
        'buy_n': len(buy_trades),
        'buy_pnl': sum(t.pnl for t in buy_trades),
        'sell_n': len(sell_trades),
        'sell_pnl': sum(t.pnl for t in sell_trades),
        'keltner_n': len(keltner),
        'keltner_pnl': sum(t.pnl for t in keltner),
        'orb_n': len(orb),
        'orb_pnl': sum(t.pnl for t in orb),
        'rsi_n': len(rsi),
        'rsi_pnl': sum(t.pnl for t in rsi),
        'trailing_n': len(trailing),
        'trailing_pnl': sum(t.pnl for t in trailing),
        'sl_n': len(sl_exits),
        'sl_pnl': sum(t.pnl for t in sl_exits),
        'timeout_n': len(timeout),
        'timeout_pnl': sum(t.pnl for t in timeout),
    }

print(f'{len(SCENARIOS)} scenarios defined')

## Part 1: 基础情景测试（当前实盘参数 + $0.50 点差）

In [ ]:
base_results = []

for sc in SCENARIOS:
    print(f"\n{'='*70}")
    print(f"{sc['id']}: {sc['name']}")
    print(f"  Period: {sc['start']} -> {sc['end']}")
    print(f"  Desc: {sc['desc']}")
    print(f"  Relevance: {sc['relevance']}")
    
    sdata = data.slice(sc['start'], sc['end'])
    print(f"  Data: M15={len(sdata.m15_df)} bars, H1={len(sdata.h1_df)} bars")
    
    if len(sdata.h1_df) < 50:
        print(f"  SKIP: insufficient data")
        base_results.append({'id': sc['id'], 'name': sc['name'], 'n': 0, 'sharpe': 0, 'total_pnl': 0, 'max_dd': 0})
        continue
    
    result = run_variant(sdata, f"{sc['id']}_base", **LIVE_KWARGS)
    trades = result.get('_trades', [])
    detail = analyze_trades(trades, sc['id'])
    
    result['id'] = sc['id']
    result['name'] = sc['name']
    result['detail'] = detail
    base_results.append(result)
    
    print(f"\n  === Results ===")
    print(f"  N={result['n']:>5}  Sharpe={result['sharpe']:>6.2f}  PnL=${result['total_pnl']:>8.0f}  MaxDD=${result['max_dd']:>6.0f}  WR={result['win_rate']:.1f}%")
    print(f"  BUY:  N={detail['buy_n']:>3}  PnL=${detail['buy_pnl']:>7.0f}")
    print(f"  SELL: N={detail['sell_n']:>3}  PnL=${detail['sell_pnl']:>7.0f}")
    print(f"  Keltner: N={detail['keltner_n']:>3} PnL=${detail['keltner_pnl']:>7.0f}  |  ORB: N={detail['orb_n']:>3} PnL=${detail['orb_pnl']:>7.0f}")
    print(f"  Exit: Trailing={detail['trailing_n']}(${detail['trailing_pnl']:.0f})  SL={detail['sl_n']}(${detail['sl_pnl']:.0f})  Timeout={detail['timeout_n']}(${detail['timeout_pnl']:.0f})")

print(f"\n{'='*70}")
print("\n=== Scenario Risk Matrix (Base) ===")
print(f"{'Scenario':35s} {'N':>5} {'Sharpe':>7} {'PnL':>9} {'MaxDD':>7} {'WR%':>6}")
print('-' * 75)
for r in base_results:
    if r['n'] > 0:
        print(f"{r.get('id','?')+': '+r.get('name','?'):35s} {r['n']:>5} {r['sharpe']:>7.2f} ${r['total_pnl']:>8.0f} ${r['max_dd']:>6.0f} {r['win_rate']:>5.1f}%")
    else:
        print(f"{r.get('id','?')+': '+r.get('name','?'):35s}   SKIP (insufficient data)")

## Part 2: 点差压力变体 — 极端行情点差扩大

In [ ]:
spread_scenarios = {
    'S1': [0.50, 0.80],          # risk-on: spread widens moderately
    'S2': [0.50, 0.80],          # crash: spread widens
    'S3': [0.50, 0.80, 1.00],    # liquidity crisis: extreme spread
    'S6': [0.50, 0.80],          # whipsaw: spread widens
}

spread_results = []

for sc in SCENARIOS:
    if sc['id'] not in spread_scenarios:
        continue
    
    sdata = data.slice(sc['start'], sc['end'])
    if len(sdata.h1_df) < 50:
        continue
    
    print(f"\n=== {sc['id']}: Spread Stress ===")
    for sp in spread_scenarios[sc['id']]:
        kw = {**LIVE_KWARGS, 'spread_cost': sp}
        r = run_variant(sdata, f"{sc['id']}_sp{sp}", verbose=False, **kw)
        spread_results.append({
            'scenario': sc['id'], 'spread': sp,
            'n': r['n'], 'sharpe': r['sharpe'],
            'pnl': r['total_pnl'], 'max_dd': r['max_dd'],
        })
        print(f"  sp=${sp:.2f}: N={r['n']:>4} Sharpe={r['sharpe']:>6.2f} PnL=${r['total_pnl']:>7.0f} MaxDD=${r['max_dd']:>6.0f}")

print("\n=== Spread Stress Summary ===")
for r in spread_results:
    print(f"  {r['scenario']} sp=${r['spread']:.2f}: Sharpe={r['sharpe']:.2f}, PnL=${r['pnl']:.0f}")

## Part 3: V3 ATR Regime 开/关对比 — 波动率自适应的情景价值

In [ ]:
v3_results = []

LIVE_NO_V3 = {k: v for k, v in LIVE_KWARGS.items() if k != 'regime_config'}

for sc in SCENARIOS:
    sdata = data.slice(sc['start'], sc['end'])
    if len(sdata.h1_df) < 50:
        continue
    
    r_v3 = run_variant(sdata, f"{sc['id']}_V3_ON", verbose=False, **LIVE_KWARGS)
    r_no = run_variant(sdata, f"{sc['id']}_V3_OFF", verbose=False, **LIVE_NO_V3)
    
    v3_results.append({
        'scenario': sc['id'], 'name': sc['name'],
        'v3_sharpe': r_v3['sharpe'], 'no_sharpe': r_no['sharpe'],
        'v3_pnl': r_v3['total_pnl'], 'no_pnl': r_no['total_pnl'],
        'v3_dd': r_v3['max_dd'], 'no_dd': r_no['max_dd'],
        'v3_n': r_v3['n'], 'no_n': r_no['n'],
    })

print("=== V3 ATR Regime: ON vs OFF per Scenario ===")
print(f"{'Scenario':30s} {'V3 Sharpe':>10} {'No Sharpe':>10} {'Delta':>7} {'V3 PnL':>8} {'No PnL':>8} {'V3 DD':>7} {'No DD':>7}")
print('-' * 95)
v3_wins = 0
for r in v3_results:
    delta = r['v3_sharpe'] - r['no_sharpe']
    winner = 'V3' if delta > 0 else 'OFF'
    if delta > 0: v3_wins += 1
    print(f"{r['scenario']+': '+r['name']:30s} {r['v3_sharpe']:>10.2f} {r['no_sharpe']:>10.2f} {delta:>+7.2f} ${r['v3_pnl']:>7.0f} ${r['no_pnl']:>7.0f} ${r['v3_dd']:>6.0f} ${r['no_dd']:>6.0f}  [{winner}]")
print(f"\nV3 wins {v3_wins}/{len(v3_results)} scenarios")

## Part 4: 仓位与门控变体 — 极端行情下的防御策略

In [ ]:
defense_tests = {
    'S2': [
        ('MaxPos=1', {'max_positions': 1}),
        ('Choppy=0.45', {'choppy_threshold': 0.45}),
    ],
    'S3': [
        ('MaxPos=1', {'max_positions': 1}),
        ('Choppy=0.50', {'choppy_threshold': 0.50}),
        ('KConly=0.65', {'kc_only_threshold': 0.65}),
    ],
    'S4': [
        ('MaxPos=1', {'max_positions': 1}),
        ('No_V3', {}),  # handled separately
    ],
    'S6': [
        ('Choppy=0.45', {'choppy_threshold': 0.45}),
        ('Choppy=0.50', {'choppy_threshold': 0.50}),
        ('MaxPos=1', {'max_positions': 1}),
    ],
}

defense_results = []

for sc_id, variants in defense_tests.items():
    sc = next(s for s in SCENARIOS if s['id'] == sc_id)
    sdata = data.slice(sc['start'], sc['end'])
    if len(sdata.h1_df) < 50:
        continue
    
    base = run_variant(sdata, f"{sc_id}_defense_base", verbose=False, **LIVE_KWARGS)
    print(f"\n=== {sc_id}: Defense Variants ===")
    print(f"  Base: N={base['n']} Sharpe={base['sharpe']:.2f} PnL=${base['total_pnl']:.0f} MaxDD=${base['max_dd']:.0f}")
    
    for vlabel, overrides in variants:
        if vlabel == 'No_V3':
            kw = {k: v for k, v in LIVE_KWARGS.items() if k != 'regime_config'}
        else:
            kw = {**LIVE_KWARGS, **overrides}
        r = run_variant(sdata, f"{sc_id}_{vlabel}", verbose=False, **kw)
        d_sharpe = r['sharpe'] - base['sharpe']
        d_pnl = r['total_pnl'] - base['total_pnl']
        d_dd = r['max_dd'] - base['max_dd']
        print(f"  {vlabel:15s}: N={r['n']:>4} Sharpe={r['sharpe']:>6.2f}({d_sharpe:>+.2f}) PnL=${r['total_pnl']:>7.0f}({d_pnl:>+.0f}) MaxDD=${r['max_dd']:>6.0f}({d_dd:>+.0f})")
        defense_results.append({
            'scenario': sc_id, 'variant': vlabel,
            'n': r['n'], 'sharpe': r['sharpe'], 'pnl': r['total_pnl'], 'max_dd': r['max_dd'],
            'delta_sharpe': d_sharpe, 'delta_pnl': d_pnl, 'delta_dd': d_dd,
        })

## Part 5: 最大单日亏损 & 连续亏损分析

In [ ]:
print("=== Worst-Case Daily & Streak Analysis per Scenario ===")
print(f"{'Scenario':30s} {'Worst Day':>10} {'Max Streak':>11} {'Streak PnL':>11} {'Avg Hold':>9}")
print('-' * 80)

daily_analysis = []

for sc in SCENARIOS:
    sdata = data.slice(sc['start'], sc['end'])
    if len(sdata.h1_df) < 50:
        continue
    
    r = run_variant(sdata, f"{sc['id']}_daily", verbose=False, **LIVE_KWARGS)
    trades = r.get('_trades', [])
    if not trades:
        continue
    
    # Worst single day
    daily_pnl = {}
    for t in trades:
        d = pd.Timestamp(t.exit_time).date()
        daily_pnl[d] = daily_pnl.get(d, 0) + t.pnl
    worst_day = min(daily_pnl.values()) if daily_pnl else 0
    best_day = max(daily_pnl.values()) if daily_pnl else 0
    
    # Max consecutive losses
    max_streak = 0
    streak = 0
    streak_pnl = 0
    worst_streak_pnl = 0
    current_streak_pnl = 0
    for t in trades:
        if t.pnl <= 0:
            streak += 1
            current_streak_pnl += t.pnl
            if streak > max_streak:
                max_streak = streak
                worst_streak_pnl = current_streak_pnl
        else:
            streak = 0
            current_streak_pnl = 0
    
    avg_hold = np.mean([t.bars_held for t in trades])
    
    print(f"{sc['id']+': '+sc['name']:30s} ${worst_day:>9.0f} {max_streak:>11} ${worst_streak_pnl:>10.0f} {avg_hold:>8.1f}")
    
    daily_analysis.append({
        'scenario': sc['id'],
        'worst_day': worst_day, 'best_day': best_day,
        'max_streak': max_streak, 'worst_streak_pnl': worst_streak_pnl,
        'avg_hold': avg_hold,
        'n_days': len(daily_pnl),
        'neg_days': sum(1 for v in daily_pnl.values() if v < 0),
    })

print("\n=== Daily PnL Distribution ===")
for da in daily_analysis:
    neg_pct = da['neg_days'] / da['n_days'] * 100 if da['n_days'] > 0 else 0
    print(f"  {da['scenario']}: {da['n_days']} trading days, {da['neg_days']} negative ({neg_pct:.0f}%), best=${da['best_day']:.0f}, worst=${da['worst_day']:.0f}")

## Part 6: 情景风险矩阵 & 综合结论

In [ ]:
print("\n" + "="*90)
print("                    SCENARIO RISK MATRIX — FINAL SUMMARY")
print("="*90)

risk_matrix = []

for sc in SCENARIOS:
    sdata = data.slice(sc['start'], sc['end'])
    if len(sdata.h1_df) < 50:
        continue
    
    r = run_variant(sdata, f"{sc['id']}_final", verbose=False, **LIVE_KWARGS)
    trades = r.get('_trades', [])
    detail = analyze_trades(trades)
    
    # Risk rating
    if r['sharpe'] > 1.0:
        risk = 'LOW'
    elif r['sharpe'] > 0:
        risk = 'MEDIUM'
    elif r['sharpe'] > -1.0:
        risk = 'HIGH'
    else:
        risk = 'EXTREME'
    
    entry = {
        'id': sc['id'], 'name': sc['name'],
        'n': r['n'], 'sharpe': r['sharpe'],
        'pnl': r['total_pnl'], 'max_dd': r['max_dd'],
        'wr': r['win_rate'],
        'buy_pnl': detail['buy_pnl'], 'sell_pnl': detail['sell_pnl'],
        'trailing_pct': detail['trailing_n'] / max(r['n'], 1) * 100,
        'risk': risk,
    }
    risk_matrix.append(entry)

print(f"\n{'ID':4s} {'Scenario':32s} {'N':>5} {'Sharpe':>7} {'PnL':>9} {'MaxDD':>7} {'WR%':>6} {'Risk':>8}")
print('-' * 85)
for e in risk_matrix:
    print(f"{e['id']:4s} {e['name']:32s} {e['n']:>5} {e['sharpe']:>7.2f} ${e['pnl']:>8.0f} ${e['max_dd']:>6.0f} {e['wr']:>5.1f}% {e['risk']:>8}")

print(f"\n=== Direction Breakdown ===")
print(f"{'Scenario':30s} {'BUY PnL':>9} {'SELL PnL':>9} {'Trail%':>7}")
print('-' * 60)
for e in risk_matrix:
    print(f"{e['id']+' '+e['name']:30s} ${e['buy_pnl']:>8.0f} ${e['sell_pnl']:>8.0f} {e['trailing_pct']:>6.1f}%")

# Worst case scenario
worst = min(risk_matrix, key=lambda x: x['pnl'])
best = max(risk_matrix, key=lambda x: x['pnl'])
print(f"\n=== Extreme Cases ===")
print(f"  BEST scenario:  {best['id']} {best['name']} — PnL ${best['pnl']:.0f}, Sharpe {best['sharpe']:.2f}")
print(f"  WORST scenario: {worst['id']} {worst['name']} — PnL ${worst['pnl']:.0f}, Sharpe {worst['sharpe']:.2f}")
print(f"  Max possible loss in any scenario: ${worst['max_dd']:.0f}")

profitable = sum(1 for e in risk_matrix if e['pnl'] > 0)
print(f"  Profitable scenarios: {profitable}/{len(risk_matrix)}")
print(f"  Avg Sharpe across scenarios: {np.mean([e['sharpe'] for e in risk_matrix]):.2f}")

In [ ]:
# Save results
save_data = {
    'risk_matrix': risk_matrix,
    'spread_results': spread_results,
    'v3_results': v3_results,
    'defense_results': defense_results,
    'daily_analysis': daily_analysis,
}

def sanitize(obj):
    if isinstance(obj, dict):
        return {k: sanitize(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [sanitize(v) for v in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

with open('../data/exp27_stress_results.json', 'w') as f:
    json.dump(sanitize(save_data), f, indent=2, default=str)

print('Results saved to data/exp27_stress_results.json')